# inference.ipynb — 최종 추론 및 제출 파일 생성

**목적**: `train.ipynb`에서 저장한 모델(`models/catboost_seed{42,7,2024}.cbm`)과 설정(`models/model_config.json`)을 불러와 2025년 테스트 기간을 예측하고, 대회 제출 규격(`src/submission.py`)에 맞는 CSV를 만든다. **외부 API 호출 없음, 이 노트북만 단독으로 처음부터 끝까지 실행 가능**(2차 평가 요건 — `ensemble-final` 스킬 5절).

**입력**: `data/processed/test_features_v2.parquet`, `models/catboost_seed*.cbm`, `models/model_config.json`, `data/sample_submission.csv`
**출력**: `submissions/YYYYMMDD_vN_설명.csv`

**주의**: 이 노트북은 로컬에 제출 파일(CSV)만 만든다. 실제 대회 플랫폼에 제출(업로드)하는 것은 별도 단계이며 민석님이 직접 하고, 일일 제출 5회 제한을 고려해 신중하게 선택한다.

## 0. 셋업

In [8]:
import sys
import json
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "data").exists(), "REPO_ROOT를 찾지 못했습니다. 노트북 실행 위치를 확인하세요."

sys.path.insert(0, str(REPO_ROOT))
from src.metric import TARGET_COLS, CAPACITY_KWH
from src.submission import validate_submission

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
MODELS_DIR = REPO_ROOT / "models"
SUBMISSIONS_DIR = REPO_ROOT / "submissions"
SUBMISSIONS_DIR.mkdir(exist_ok=True)

print("python executable:", sys.executable)
print("REPO_ROOT:", REPO_ROOT)

python executable: d:\공모전\wind_forecast\venv\Scripts\python.exe
REPO_ROOT: d:\공모전\wind_forecast


## 1. 설정·모델 로드

`train.ipynb`가 저장한 `model_config.json`을 그대로 읽어 피처 목록·시드·손실 함수 등을 복원한다. 값을 이 노트북에 다시 하드코딩하지 않는 이유: 두 노트북 사이에 설정이 어긋나면(예: 피처 목록이 조금이라도 다르면) 조용히 틀린 예측이 나올 수 있기 때문 — 파일로 한 번만 정의하고 양쪽이 그 파일을 참조하는 게 안전하다.

In [9]:
with open(MODELS_DIR / "model_config.json", encoding="utf-8") as f:
    model_config = json.load(f)

FEATURE_COLS = model_config["feature_cols"]
GROUP_ID_MAP = model_config["group_id_map"]
ENSEMBLE_SEEDS = model_config["ensemble_seeds"]
GROUP_ID_CATEGORIES = list(GROUP_ID_MAP.values())

models = {}
for seed in ENSEMBLE_SEEDS:
    model = CatBoostRegressor()
    model.load_model(str(MODELS_DIR / f"catboost_seed{seed}.cbm"))
    models[seed] = model

print("불러온 모델 시드:", list(models.keys()))
print("피처 개수:", len(FEATURE_COLS))
print("quantile_alpha:", model_config["quantile_alpha"], "| use_sample_weight:", model_config["use_sample_weight"])

불러온 모델 시드: [42, 7, 2024]
피처 개수: 50
quantile_alpha: 0.7 | use_sample_weight: True


## 2. 테스트 데이터 로드

In [10]:
test_features = pd.read_parquet(PROCESSED_DIR / "test_features_v2.parquet")
print("test_features:", test_features.shape)
print("기간:", test_features["forecast_kst_dtm"].min(), "~", test_features["forecast_kst_dtm"].max())

missing_cols = [c for c in FEATURE_COLS if c not in test_features.columns]
assert not missing_cols, f"train에는 있는데 test에는 없는 피처: {missing_cols}"
print("train.ipynb가 쓴 피처가 test에 모두 존재함을 확인")

test_features: (8760, 52)
기간: 2025-01-01 01:00:00 ~ 2026-01-01 00:00:00
train.ipynb가 쓴 피처가 test에 모두 존재함을 확인


## 3. 그룹별 예측 — seed 평균 앙상블

`train.ipynb`와 동일하게 그룹마다 `group_id`를 고정해 예측한 뒤, 이용률→kWh로 환산하고 `[0, 설비용량]`으로 클리핑한다. 앙상블은 seed별 예측을 평균(`ensemble-final` 스킬 3절).

In [11]:
features = FEATURE_COLS + ["group_id"]
predictions = {}

for g in TARGET_COLS:
    g_rows = test_features.copy()
    g_rows["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(g_rows), categories=GROUP_ID_CATEGORIES)

    seed_preds = []
    for seed in ENSEMBLE_SEEDS:
        util_pred = models[seed].predict(g_rows[features])
        seed_preds.append(util_pred * CAPACITY_KWH[g])

    ensembled = np.mean(seed_preds, axis=0)
    predictions[g] = np.clip(ensembled, 0, CAPACITY_KWH[g])
    print(f"{g}: 클리핑 전 범위 [{ensembled.min():.1f}, {ensembled.max():.1f}], "
          f"클리핑 후 [{predictions[g].min():.1f}, {predictions[g].max():.1f}]")

kpx_group_1: 클리핑 전 범위 [-388.4, 22020.5], 클리핑 후 [0.0, 21600.0]
kpx_group_2: 클리핑 전 범위 [-361.2, 22033.0], 클리핑 후 [0.0, 21600.0]
kpx_group_3: 클리핑 전 범위 [-836.8, 21365.7], 클리핑 후 [0.0, 21000.0]


## 4. 제출 형식으로 조립

`sample_submission.csv`의 행 순서(`forecast_id`)를 기준으로 병합해서, 순서가 어긋날 가능성을 원천 차단한다(`src/submission.py`가 순서까지 검사함).

In [12]:
sample_submission = pd.read_csv(
    REPO_ROOT / "data" / "sample_submission.csv",
    encoding="utf-8-sig",
    dtype={"forecast_id": str},
    parse_dates=["forecast_kst_dtm"],
)

pred_df = test_features[["forecast_id", "forecast_kst_dtm"]].copy()
for g in TARGET_COLS:
    pred_df[g] = predictions[g]

submission = sample_submission[["forecast_id", "forecast_kst_dtm"]].merge(
    pred_df, on=["forecast_id", "forecast_kst_dtm"], how="left",
)

assert submission[TARGET_COLS].isna().sum().sum() == 0, "병합 후 결측 발생 — forecast_id/시각 불일치 가능성"
print(submission.shape)
submission.head(3)

(8760, 5)


,forecast_id,forecast_kst_dtm,kpx_group_1,kpx_group_2,kpx_group_3
0,forecast_0001,2025-01-01 01:00:00,19869.859226,20041.125165,18745.834073
1,forecast_0002,2025-01-01 02:00:00,19766.773441,20027.611686,18413.111903
2,forecast_0003,2025-01-01 03:00:00,18770.973545,19093.695338,17438.940949


## 5. Sanity check — 예측이 그럴듯한가

`ensemble-final` 스킬 4절: 재학습 모델은 검증 점수가 없으므로, 예측 분포가 과거(2024) 패턴과 형태가 비슷한지로 명백한 오류만 확인한다. 2025년의 실제 날씨가 2024년과 정확히 같을 수는 없으니 **완전히 같은 모양일 필요는 없고, 계절 패턴(겨울↑여름↓ 등)이 뒤집히거나 극단적으로 어긋나지 않는지**를 본다.

In [13]:
train_features_for_compare = pd.read_parquet(PROCESSED_DIR / "train_features_v2.parquet")
train_2024 = train_features_for_compare[train_features_for_compare["forecast_kst_dtm"].dt.year == 2024].copy()
train_2024["month"] = train_2024["forecast_kst_dtm"].dt.month

submission_compare = submission.copy()
submission_compare["month"] = submission_compare["forecast_kst_dtm"].dt.month

monthly_2024 = train_2024.groupby("month")[TARGET_COLS].mean()
monthly_2025_pred = submission_compare.groupby("month")[TARGET_COLS].mean()

monthly_compare = monthly_2024.join(monthly_2025_pred, lsuffix="_actual_2024", rsuffix="_pred_2025")
print("월별 평균 발전량(kWh) — 2024년 실제 vs 2025년 예측:")
print(monthly_compare)

for g in TARGET_COLS:
    at_zero = (submission[g] <= 1e-6).mean()
    at_capacity = (submission[g] >= CAPACITY_KWH[g] - 1e-6).mean()
    print(f"{g}: 0kWh 근접 비율={at_zero:.1%}, 설비용량 근접 비율={at_capacity:.1%}")

월별 평균 발전량(kWh) — 2024년 실제 vs 2025년 예측:
       kpx_group_1_actual_2024  kpx_group_2_actual_2024  \
month                                                     
1                  8454.765524              8120.868776   
2                  4767.522322              5026.487644   
3                  7976.937009              7740.666558   
4                  4649.503486              4805.846233   
5                  7884.598289              7909.422819   
6                  3872.189119              4082.668246   
7                  9162.998454              9709.697989   
8                  3026.687923              3190.564832   
9                  2644.849539              3061.348313   
10                 3974.258054              3942.182497   
11                 5838.134021              5954.944110   
12                13039.120046             13987.720469   

       kpx_group_3_actual_2024  kpx_group_1_pred_2025  kpx_group_2_pred_2025  \
month                                                 

### 5-1. 월별 차이가 큰 이유 — 입력 풍속 자체가 다른지 확인

위 표를 보면 일부 달(2월·4월·6월·8월·9월·11월)은 예측이 2024년 실제보다 80~180%나 높고, 반대로 7월은 오히려 낮다. 이 정도로 큰 차이가 파이프라인 버그 때문인지, 아니면 "2025년 예보의 풍속 자체가 그 정도로 다르기 때문"인지 구분해야 한다. `wind-domain-features` 스킬 1절: **발전량은 풍속의 세제곱(v³)에 비례**하므로, 풍속이 조금만 달라도(예: 1.2배) 발전량 예측은 훨씬 크게(1.2³≈1.7배) 벌어질 수 있다 — 그러니 먼저 입력 풍속 자체를 연도별로 비교해서, 발전량 차이가 "풍속 차이의 자연스러운 증폭"으로 설명되는지 본다.

In [14]:
test_features_month = test_features.copy()
test_features_month["month"] = test_features_month["forecast_kst_dtm"].dt.month
train_features_for_compare["month"] = train_features_for_compare["forecast_kst_dtm"].dt.month
train_features_for_compare["year"] = train_features_for_compare["forecast_kst_dtm"].dt.year

wind_cols = ["gfs_ws100m", "kpx_group_1_ldaps_ws10m", "kpx_group_2_ldaps_ws10m", "kpx_group_3_ldaps_ws10m"]

wind_2024 = train_features_for_compare[train_features_for_compare["year"] == 2024].groupby("month")[wind_cols].mean()
wind_2025 = test_features_month.groupby("month")[wind_cols].mean()

wind_ratio = wind_2025[wind_cols] / wind_2024[wind_cols]
wind_ratio.columns = [f"{c}_2025/2024_ratio" for c in wind_cols]
print("풍속 비율(2025 예보 / 2024 실제, 1보다 크면 2025이 더 셈):")
wind_ratio

풍속 비율(2025 예보 / 2024 실제, 1보다 크면 2025이 더 셈):


,gfs_ws100m_2025/2024_ratio,kpx_group_1_ldaps_ws10m_2025/2024_ratio,kpx_group_2_ldaps_ws10m_2025/2024_ratio,kpx_group_3_ldaps_ws10m_2025/2024_ratio
month,,,,
1,1.416432,0.984764,0.998349,1.014259
2,1.803708,1.308924,1.399153,1.501640
3,0.951351,0.941501,0.942679,0.943522
4,1.868146,1.327292,1.346851,1.403293
5,1.019231,1.015951,0.985976,0.955227
6,1.329547,1.198119,1.196612,1.197879
7,0.586282,0.692582,0.706719,0.708369
8,1.253062,1.195015,1.262214,1.328352
9,1.170748,1.205745,1.194108,1.187842


## 6. 제출 파일 저장 + 검증

파일명 규칙(`CLAUDE.md` 8절): `submissions/YYYYMMDD_vN_설명.csv`. **`validate_submission()` 통과가 제출 전 필수 조건**이다. `description`은 이 예측의 핵심 설정을 요약해 나중에 파일 이름만 보고도 어떤 모델인지 알 수 있게 한다.

In [15]:
today_str = date.today().strftime("%Y%m%d")
version = "v1"
description = "catboost_quantile070_actualweight_seedavg"
submission_path = SUBMISSIONS_DIR / f"{today_str}_{version}_{description}.csv"

submission.to_csv(submission_path, index=False, encoding="utf-8-sig")
print(f"저장: {submission_path}")

validate_submission(submission_path)

저장: d:\공모전\wind_forecast\submissions\20260722_v1_catboost_quantile070_actualweight_seedavg.csv
[PASS] 20260722_v1_catboost_quantile070_actualweight_seedavg.csv: 컬럼/행 수/ID 불변/결측·음수·상한 초과 없음 확인 완료 (8760행)


True

## 요약

- `train.ipynb`가 저장한 seed 3개 모델(42/7/2024)과 `model_config.json`을 불러와 2025년 테스트 기간을 예측했다
- 그룹별로 예측 → seed 평균 앙상블 → `[0, 설비용량]` 클리핑
- `sample_submission.csv`의 `forecast_id` 순서에 병합해서 순서 불일치를 원천 차단했다
- 2024년 월별 실제 발전량과 비교하는 sanity check로 명백한 오류가 없는지 확인했다
- `validate_submission()`을 통과한 제출 파일을 `submissions/`에 저장했다 — **실제 대회 플랫폼 제출은 민석님이 직접 진행**하고, 일일 5회 제한을 고려해 신중하게 선택

**다음**: 실제 제출 후 리더보드 점수를 `HANDOFF.md`에 기록(`session-handoff` 스킬)하고, `reports/05_tuning.md` 10절 오류 분석에서 제안한 group_3 전용 대응·중간 풍속 구간 피처 보강을 다음 실험으로 검토